# Global MMLU preprocessing

This notebook loads Global MMLU for Russian, Chinese, Italian, Spanish, and Hindi; explores cultural-sensitivity labels and subject categories; cleans and deduplicates MCQ records; maps `cultural_sensitivity_label` into our `regionality` labels; applies fixed-seed subject-category sampling; and saves SFT-compatible JSONL files under `multilingual/data/processed/global_mmlu/`.

In [ ]:
from pathlib import Path
import json
import os
import random

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

In [ ]:
REPO_ROOT = Path('/scratch/workspace/emma/standard-project-m2-mystery-machine')
OUTPUT_DIR = REPO_ROOT / 'multilingual/data/processed/global_mmlu'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'CohereLabs/Global-MMLU'
LANGUAGE_CONFIGS = {
    'russian': 'ru',
    'chinese': 'zh',
    'italian': 'it',
    'spanish': 'es',
    'hindi': 'hi',
}
SOURCE_SPLIT = 'test'
RANDOM_SEED = 42
VALID_ANSWERS = {'A', 'B', 'C', 'D'}
VALID_REGIONALITIES = {'regional', 'regional_implicit', 'region_agnostic'}
REGIONALITY_MAP = {'CS': 'regional', 'CC': 'regional_implicit'}

# Adjust after inspecting the exact subject_category names below.
SUBJECT_CATEGORY_WEIGHTS = {
    'STEM': 0.3,
    'Natural Science': 0.3,
    'Natural Sciences': 0.3,
    'Humanities': 1.0,
    'Social Sciences': 1.0,
    'Business & Economics': 1.0,
    'Business and Economics': 1.0,
    'Law': 1.0,
    'Medical & Health': 1.0,
    'Medicine & Health': 1.0,
    'Health': 1.0,
    'Other': 0.5,
}
DEFAULT_CATEGORY_WEIGHT = 0.5
HF_TOKEN = os.environ.get('HF_TOKEN')

## Load all languages

In [ ]:
frames = []

for language, config in LANGUAGE_CONFIGS.items():
    print(f'Loading {DATASET_NAME} | config={config} | split={SOURCE_SPLIT}')
    ds = load_dataset(DATASET_NAME, config, split=SOURCE_SPLIT, token=HF_TOKEN)
    df_lang = ds.to_pandas()
    df_lang['language'] = language
    df_lang['source_split'] = SOURCE_SPLIT
    df_lang['source_idx'] = range(len(df_lang))
    frames.append(df_lang)

df_raw = pd.concat(frames, ignore_index=True)
print(f'Loaded rows: {len(df_raw)}')
display(df_raw.head())

## Inspect labels and categories before filtering

In [ ]:
print('Columns:')
print(df_raw.columns.tolist())

print('
cultural_sensitivity_label values:')
print(sorted(df_raw['cultural_sensitivity_label'].dropna().astype(str).unique().tolist()))

print('
subject_category values:')
subject_categories = sorted(df_raw['subject_category'].dropna().astype(str).unique().tolist())
for c in subject_categories:
    print(repr(c))

missing_weights = [c for c in subject_categories if c not in SUBJECT_CATEGORY_WEIGHTS]
print('
Subject categories using DEFAULT_CATEGORY_WEIGHT:', missing_weights)

In [ ]:
print('Counts per language × cultural_sensitivity_label:')
display(pd.crosstab(df_raw['language'], df_raw['cultural_sensitivity_label'], dropna=False))

print('
Counts per subject_category:')
display(df_raw['subject_category'].value_counts(dropna=False).sort_index().to_frame('n_examples'))

print('
Top 30 subjects:')
display(df_raw['subject'].value_counts(dropna=False).head(30).to_frame('n_examples'))

In [ ]:
plot_df = pd.crosstab(df_raw['language'], df_raw['cultural_sensitivity_label'], dropna=False)
ax = plot_df.plot(kind='bar', figsize=(10, 5))
ax.set_title('Global MMLU examples per language and cultural_sensitivity_label')
ax.set_xlabel('Language')
ax.set_ylabel('Number of examples')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Clean records

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()


def clean_answer(x):
    if pd.isna(x):
        return None
    ans = str(x).strip().upper()
    return ans if ans in VALID_ANSWERS else None


def map_regionality(x):
    if pd.isna(x):
        return 'region_agnostic'
    return REGIONALITY_MAP.get(str(x).strip(), 'region_agnostic')


def make_prompt(row):
    return (
        f"{clean_text(row['question'])}

"
        f"A) {clean_text(row['option_a'])}
"
        f"B) {clean_text(row['option_b'])}
"
        f"C) {clean_text(row['option_c'])}
"
        f"D) {clean_text(row['option_d'])}"
    )


df = df_raw.copy()
for col in ['question', 'option_a', 'option_b', 'option_c', 'option_d']:
    df[col] = df[col].map(clean_text)

df['answer'] = df['answer'].map(clean_answer)
df['regionality'] = df['cultural_sensitivity_label'].map(map_regionality)

n_before = len(df)
df = df[df['question'].ne('')]
df = df[df['option_a'].ne('') & df['option_b'].ne('')]
df = df[df['answer'].isin(VALID_ANSWERS)]
df = df[~((df['answer'] == 'C') & df['option_c'].eq(''))]
df = df[~((df['answer'] == 'D') & df['option_d'].eq(''))]
df = df[df['regionality'].isin(VALID_REGIONALITIES)]

print(f'Rows before cleaning: {n_before}')
print(f'Rows after cleaning:  {len(df)}')

In [ ]:
dedup_cols = ['language', 'question', 'option_a', 'option_b', 'option_c', 'option_d']
n_before_dedup = len(df)
df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
print(f'Rows before deduplication: {n_before_dedup}')
print(f'Rows after deduplication:  {len(df)}')

## Apply subject-category sampling

All `CS` examples are kept regardless of subject category. Other examples are sampled by the fixed weights defined above.

In [ ]:
def category_weight(category):
    return SUBJECT_CATEGORY_WEIGHTS.get(clean_text(category), DEFAULT_CATEGORY_WEIGHT)


rng = random.Random(RANDOM_SEED)
keep_mask = []
for _, row in df.iterrows():
    if row['regionality'] == 'regional':
        keep_mask.append(True)
    else:
        keep_mask.append(rng.random() < category_weight(row.get('subject_category', '')))

df_before_sampling = df.copy()
df = df.loc[keep_mask].reset_index(drop=True)

print(f'Rows before subject-category sampling: {len(df_before_sampling)}')
print(f'Rows after subject-category sampling:  {len(df)}')

print('
Kept counts by language × regionality:')
display(pd.crosstab(df['language'], df['regionality']))

print('
Kept counts by subject_category:')
display(df['subject_category'].value_counts(dropna=False).sort_index().to_frame('n_examples'))

In [ ]:
plot_df = pd.crosstab(df['language'], df['regionality'])
ax = plot_df.plot(kind='bar', figsize=(10, 5))
ax.set_title('Clean/sampled Global MMLU examples per language and regionality')
ax.set_xlabel('Language')
ax.set_ylabel('Number of examples')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Convert to SFT-compatible records

In [ ]:
def safe_id_part(x):
    return str(x).strip().replace(' ', '_').replace('/', '_')


def make_record(row, idx):
    language = row['language']
    sample_id = clean_text(row.get('sample_id', '')) or str(row.get('source_idx', idx))
    return {
        'id': f'global_mmlu_{language}_{safe_id_part(sample_id)}',
        'dataset': 'global_mmlu',
        'language': language,
        'split': 'all',
        'task_type': 'mcq',
        'prompt': make_prompt(row),
        'answer': row['answer'],
        'metadata': {
            'subject': clean_text(row.get('subject', '')),
            'subject_category': clean_text(row.get('subject_category', '')),
            'cultural_sensitivity_label': None if pd.isna(row.get('cultural_sensitivity_label')) else clean_text(row.get('cultural_sensitivity_label')),
            'regionality': row['regionality'],
            'region': clean_text(row.get('region', '')),
            'country': clean_text(row.get('country', '')),
            'sample_id': sample_id,
            'source_split': SOURCE_SPLIT,
        },
    }


records = [make_record(row, idx) for idx, row in df.iterrows()]
print(f'Built records: {len(records)}')
records[0]

## Save JSONL files

In [ ]:
def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '
')


def read_jsonl(path):
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


for language in LANGUAGE_CONFIGS:
    language_records = [r for r in records if r['language'] == language]
    all_path = OUTPUT_DIR / f'global_mmlu_{language}_all.jsonl'
    write_jsonl(language_records, all_path)
    print(f'Saved {len(language_records)} records to {all_path}')

    regional_records = [r for r in language_records if r['metadata']['regionality'] == 'regional']
    path = OUTPUT_DIR / f'global_mmlu_{language}_regional.jsonl'
    write_jsonl(regional_records, path)
    print(f'Saved {len(regional_records)} records to {path}')

    implicit_records = [r for r in language_records if r['metadata']['regionality'] == 'regional_implicit']
    path = OUTPUT_DIR / f'global_mmlu_{language}_regional_implicit.jsonl'
    write_jsonl(implicit_records, path)
    print(f'Saved {len(implicit_records)} records to {path}')

## Save preprocessing metadata

In [ ]:
metadata = {
    'dataset_name': DATASET_NAME,
    'source_split': SOURCE_SPLIT,
    'language_configs': LANGUAGE_CONFIGS,
    'random_seed': RANDOM_SEED,
    'subject_category_weights': SUBJECT_CATEGORY_WEIGHTS,
    'default_category_weight': DEFAULT_CATEGORY_WEIGHT,
    'n_raw': len(df_raw),
    'n_after_cleaning_before_sampling': len(df_before_sampling),
    'n_after_sampling': len(df),
}
meta_path = OUTPUT_DIR / 'global_mmlu_preprocessing_metadata.json'
meta_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved metadata to {meta_path}')

## Validate saved files

In [ ]:
def validate_record(record):
    assert record['dataset'] == 'global_mmlu'
    assert record['language'] in LANGUAGE_CONFIGS
    assert record['split'] == 'all'
    assert record['task_type'] == 'mcq'
    assert record['answer'] in VALID_ANSWERS
    assert record['prompt'].strip()
    assert '

A) ' in record['prompt']
    assert record['metadata']['regionality'] in VALID_REGIONALITIES
    assert record['metadata']['source_split'] == SOURCE_SPLIT


for path in sorted(OUTPUT_DIR.glob('global_mmlu_*.jsonl')):
    loaded = read_jsonl(path)
    ids = [r['id'] for r in loaded]
    prompts = [r['prompt'] for r in loaded]
    assert len(ids) == len(set(ids)), f'Duplicate ids in {path}'
    assert len(prompts) == len(set(prompts)), f'Duplicate prompts in {path}'
    for record in loaded:
        validate_record(record)
    print(f'Valid: {path.name} ({len(loaded)} records)')

## Final summary

In [ ]:
loaded_records = []
for language in LANGUAGE_CONFIGS:
    loaded_records.extend(read_jsonl(OUTPUT_DIR / f'global_mmlu_{language}_all.jsonl'))

summary_df = pd.DataFrame({
    'language': r['language'],
    'answer': r['answer'],
    'regionality': r['metadata']['regionality'],
    'subject': r['metadata']['subject'],
    'subject_category': r['metadata']['subject_category'],
    'cultural_sensitivity_label': r['metadata']['cultural_sensitivity_label'],
} for r in loaded_records)

print('Final counts by language × regionality:')
display(pd.crosstab(summary_df['language'], summary_df['regionality']))

print('
Final counts by subject_category:')
display(summary_df['subject_category'].value_counts(dropna=False).sort_index().to_frame('n_examples'))

print('
Top 30 final subjects:')
display(summary_df['subject'].value_counts(dropna=False).head(30).to_frame('n_examples'))

print(f'Output directory: {OUTPUT_DIR}')